## Qwen/Qwen2.5-14B-Instruct-AWQ

In [1]:
print("test jupyter")

test jupyter


In [2]:
%pip install transformers torch==2.5 accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.5/906.5 MB 226.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 135.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 353.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 454.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 238.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 382.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 353.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 285.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 312.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 383.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 360.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 5

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

ModuleNotFoundError: No module named 'torch'

In [4]:
# We pull a version that has already been compressed to 4-bit by the community
model_id = "Qwen/Qwen2.5-14B-Instruct"

print("Downloading pre-quantized blocks directly to GPU...")

# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Stream the 4-bit files straight into your hardware
# This will bypass your 12GB system RAM limitation and fit easily into your 16GB VRAM
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="cuda" # Automatically targets your GPU
)

print("Model successfully loaded onto your GPU! Ready to chat.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model successfully loaded onto your GPU! Ready to chat.


In [9]:
# 3. Use the model immediately
messages = [
    {"role": "system", "content": "Translate the following English social-media text into natural Indonesian. Preserve the original meaning, tone, ambiguity, slang, exaggeration, sarcasm, and pragmatic cues as closely as possible. Do not explain the text, resolve ambiguity, infer unstated meaning, or add information. Return only the translated text."},
    {"role": "user", "content": "What a successful toast, it looks so delicious!"}
]

# Format prompt using Qwen's specific template structure
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate response
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=False
)

# Extract only the newly generated text fragments
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
print("\n--- Model Response ---")
print(response[0])



--- Model Response ---
Toste itu berhasil banget nih, terlihat enak sekali!


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-14B-Instruct-AWQ"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


## Deepseek-r1:14b

In [1]:
import ollama

In [ ]:
model_name = "deepseek-r1:14b"

stream = ollama.chat(
    model=model_name,
    messages=[
    {"role": "system", "content": "Translate the following English social-media text into natural Indonesian. Preserve the original meaning, tone, ambiguity, slang, exaggeration, sarcasm, and pragmatic cues as closely as possible. Do not explain the text, resolve ambiguity, infer unstated meaning, or add information. Return only the translated text."},
    {"role": "user", "content": "What a successful toast, it looks so delicious!"},
    {"role": "user", "content": "You must be fun at parties"},
    {"role": "user", "content": "What a great idea, even the undead will be dreadful"}
    ],
    stream=True,
    options={'temperature': 0}
)

# Print tokens as they arrive from the local server
for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)




Apa itu toast yang sukses, terlihat sangatlezat!  
Kamu pasti menyenangkan di pesta.  
Apa ide yang bagus! bahkan zombie pun akan merasa takut.